# RF-DETR Small — weed/crop training on Colab GPU

**Why this notebook exists:** local training on Apple-Silicon **MPS stalled hard** — RF-DETR is Deformable-DETR-based, and its `scatter_add` ops are pathologically slow on Metal (a single step hangs for minutes inside one synchronous Metal dispatch). A CPU fallback works but is multi-day slow (one epoch > 1 hour). This trains the *same* model on a **free Colab T4 GPU** in ~30 min — the way Roboflow's hosted trainer does — using the open-source `rfdetr` package (Apache-2.0). **$0 and ToS-clean.**

**On-device target:** we try **CoreML first** (cleanest iOS integration), with **ONNX as the reliable fallback** — because rfdetr has no native CoreML export (see the bottom note).

**Setup:** Runtime > Change runtime type > **T4 GPU**, then run top to bottom.

## 1. Verify the GPU

In [ ]:
!nvidia-smi
import torch
print('torch', torch.__version__, '| cuda available:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'No GPU - set Runtime > Change runtime type > T4 GPU'
print('device:', torch.cuda.get_device_name(0))

## 2. Install rfdetr (training + ONNX export) + the Roboflow SDK

The `[train,loggers]` extras are required for training (they pull in `pytorch_lightning`); `[onnx]` adds ONNX export. If pip changes the pre-installed torch, use Runtime > Restart session, then re-run from here.

In [ ]:
%pip install -q "rfdetr[train,loggers,onnx]" roboflow

## 3. Roboflow API key

Uses Colab Secrets if you've added `ROBOFLOW_API_KEY` (key icon, left sidebar); otherwise prompts. The key is never written into the notebook.

In [ ]:
API_KEY = None
try:
    from google.colab import userdata
    API_KEY = userdata.get('ROBOFLOW_API_KEY')
except Exception:
    pass
if not API_KEY:
    import getpass
    API_KEY = getpass.getpass('Roboflow API key: ')
print('key loaded:', bool(API_KEY))

## 4. Download the dataset (COCO)

The same fork used for the hosted run. Dataset download is free; only the *hosted* CoreML/weights export is paywalled — which is why we train here.

> If your dataset version isn't `1`, change `.version(...)`.

In [ ]:
from roboflow import Roboflow
rf = Roboflow(api_key=API_KEY)
project = rf.workspace("iacomus").project("weed-crop-aerial-mbyst")
dataset = project.version(1).download("coco")
DATASET_DIR = dataset.location
print('dataset at:', DATASET_DIR)
!ls -1 "$DATASET_DIR" 

## 5. Train RF-DETR Small

Same config as the local script, batch sized up for the T4 (16 GB). **If you hit CUDA OOM, drop `batch_size` to 4 and raise `grad_accum_steps` to 4.**

The 50-epoch cap + early stopping (patience 10) brackets the expected ~epoch-18 peak — the hosted run peaked there at **0.7747 mAP@50** then overfit, so longer is not better.

In [ ]:
import time
from rfdetr import RFDETRSmall

OUTPUT_DIR = 'output'
model = RFDETRSmall()
t0 = time.time()
model.train(
    dataset_dir=DATASET_DIR,
    output_dir=OUTPUT_DIR,
    epochs=50,
    batch_size=8,          # T4 16 GB; drop to 4 if you hit CUDA OOM
    grad_accum_steps=2,    # effective batch 16
    checkpoint_interval=5,
    early_stopping=True,
    early_stopping_patience=10,
    early_stopping_min_delta=0.001,
    num_workers=2,
    tensorboard=True,
)
print(f'training wall-clock: {(time.time()-t0)/60:.1f} min')

## 6. Find the best checkpoint

Note the exact filename and use it in the export cells below (usually `checkpoint_best_ema.pth`).

In [ ]:
!ls -lh output/*.pth

## 7. Export to CoreML (primary target - experimental)

rfdetr has **no native CoreML export**, and `coremltools` dropped the ONNX path - so this does a **direct PyTorch -> CoreML** trace of the underlying module (`model.model.model`), at the training resolution. Deformable-attention / dynamic ops **may** block the trace; if this cell errors, that's an expected outcome - the ONNX fallback (cell 8) still gives you a deployable artifact.

Notes for the Swift side:
- The model outputs raw `pred_boxes` + `pred_logits`; decode app-side (sigmoid, top-k, cxcywh -> xyxy).
- Input is **not** normalized inside the model - apply ImageNet mean `[0.485,0.456,0.406]` / std `[0.229,0.224,0.225]` to the RGB tensor before inference.

In [ ]:
%pip install -q coremltools
import copy, torch, coremltools as ct
from rfdetr import RFDETRSmall

BEST = 'output/checkpoint_best_ema.pth'   # adjust to the filename from step 6
m = RFDETRSmall(pretrain_weights=BEST)

# self.model.model is the underlying nn.Module (confirmed in rfdetr/detr.py);
# export uses a deepcopy in eval mode and a (1,3,R,R) dummy at model.resolution.
net = copy.deepcopy(m.model.model).eval().to('cpu')
R = int(getattr(m.model, 'resolution', 512))
print('input resolution:', R)

dummy = torch.randn(1, 3, R, R)
with torch.no_grad():
    raw = net(dummy)
print('raw forward output:', type(raw), list(raw.keys()) if isinstance(raw, dict) else 'n/a')

class Wrap(torch.nn.Module):
    def __init__(self, net):
        super().__init__()
        self.net = net
    def forward(self, x):
        o = self.net(x)
        return o['pred_boxes'], o['pred_logits']

try:
    w = Wrap(net).eval()
    traced = torch.jit.trace(w, dummy, strict=False)
    mlmodel = ct.convert(
        traced,
        inputs=[ct.TensorType(name='input', shape=(1, 3, R, R))],
        minimum_deployment_target=ct.target.iOS16,
        convert_to='mlprogram',
    )
    mlmodel.save('rfdetr_small.mlpackage')
    print('CoreML OK -> rfdetr_small.mlpackage  (inspect output names in Xcode)')
except Exception:
    import traceback
    traceback.print_exc()
    print('CoreML conversion failed (expected if deformable-attention ops are unsupported).')
    print('Use the ONNX fallback in cell 8 - that is your deployable artifact.')

## 8. Export to ONNX (reliable fallback - always run)

rfdetr's native export. Output: `export/inference_model.onnx` at the training resolution (512x512). Runs on iOS via ONNX Runtime Mobile.

> Heads-up: ONNX export has hit a `torch.export` tracing bug on some torch versions ([rf-detr#473](https://github.com/roboflow/rf-detr/issues/473)). If it errors, you still have the `.pth` from step 6 - download it (cell 10) and export later on a pinned torch.

In [ ]:
from rfdetr import RFDETRSmall
BEST = 'output/checkpoint_best_ema.pth'   # adjust to the filename from step 6
m_onnx = RFDETRSmall(pretrain_weights=BEST)
m_onnx.export(output_dir='export')        # -> export/inference_model.onnx
!ls -lh export/

## 9. (Optional) TFLite export - another native on-device format

Unlike CoreML, TFLite **is** a first-class rfdetr export and runs on iOS via LiteRT/TensorFlow-Lite. Produces fp32 + fp16 `.tflite`. Needs the `tflite` extra.

In [ ]:
# %pip install -q "rfdetr[train,loggers,onnx,tflite]"   # uncomment if you want TFLite
# m_onnx.export(output_dir='export_tflite', format='tflite')
# !ls -lh export_tflite/

## 10. Download artifacts

Bundles whatever exported successfully — `.mlpackage` (if CoreML worked), `inference_model.onnx`, and the `.pth` checkpoint.

In [ ]:
import shutil, os
from google.colab import files
os.makedirs('artifacts', exist_ok=True)
for f in ['output/checkpoint_best_ema.pth', 'export/inference_model.onnx']:
    if os.path.exists(f):
        shutil.copy(f, 'artifacts/')
if os.path.isdir('rfdetr_small.mlpackage'):
    dst = 'artifacts/rfdetr_small.mlpackage'
    shutil.rmtree(dst, ignore_errors=True)
    shutil.copytree('rfdetr_small.mlpackage', dst)
shutil.make_archive('rfdetr_weedcrop', 'zip', 'artifacts')
print('zipped:', round(os.path.getsize('rfdetr_weedcrop.zip')/1e6, 1), 'MB')
files.download('rfdetr_weedcrop.zip')

## iOS deployment — the CoreML reality (read before integrating)

Double-checking the docs surfaced a gap worth being explicit about:

- **rfdetr exports ONNX and TFLite, but NOT CoreML** ([export docs](https://rfdetr.roboflow.com/develop/learn/export/)).
- **coremltools dropped the ONNX path** (frozen ~v6), so you can't convert `inference_model.onnx` -> `.mlpackage`; modern coremltools converts from a *traced PyTorch* model (what cell 7 attempts directly).
- Tracing a **deformable-attention transformer** for CoreML is fragile (custom ops) — not guaranteed.

**Chosen path: CoreML-first, ONNX-fallback.**

| Option | Path | Trade-off |
|---|---|---|
| **CoreML** (cell 7) | PyTorch trace -> `coremltools` | Cleanest iOS integration *if* it converts; reuses the existing CoreML-based app |
| **ONNX Runtime Mobile** (cell 8) | `inference_model.onnx` -> onnxruntime-swift | Reliable export; custom Swift inference + RF-DETR decode |
| **TFLite / LiteRT** (cell 9) | rfdetr `format='tflite'` -> TF-Lite iOS | Native export; mature iOS runtime; custom glue |

This is itself an SA finding: *the free/open-source stack gets you a trained model + a portable export, but the polished CoreML + Swift-SDK path is part of what the paid platform sells.*